# Lecture 1.6 — Your First Agent: Hello World in 10 Lines
**OpenAI Agents SDK — Complete Course**  
Section 01: Introduction & Setup

---

> **Goal:** Write and run your first real agent — an `Agent` object powered by `Runner.run()` — and watch it return a live LLM response.

**What we cover in this notebook:**
1. Install / confirm the SDK
2. Import `Agent` and `Runner`
3. Define the agent
4. Run the agent with `await Runner.run()`
5. Print `final_output`
6. Inspect the `RunResult` object
7. Change the instructions — show the agent is configurable

## Cell 1 — Install / Confirm the SDK

This cell installs the `openai-agents` package, pinned to a specific version. If that exact version is already present in your environment, pip skips the install and moves on instantly. If a different version is installed, or none at all, pip installs the pinned version so this notebook behaves exactly as it was built and recorded.

The `-q` flag suppresses verbose dependency output and keeps the cell output clean.

In [ ]:
!pip install openai-agents==0.18.3 -q

## Cell 2 — Set the API Key (Google Colab)

The SDK reads your OpenAI API key from the `OPENAI_API_KEY` environment variable. The cleanest way to supply this in Google Colab is via **Colab Secrets** — no key ever appears in your notebook code or output.

### How to add your key to Colab Secrets:

1. Click the **🔑 key icon** in the left sidebar (or go to **Tools → Secrets**).
2. Click **+ Add new secret**.
3. Set the name to `OPENAI_API_KEY` and paste your key as the value.
4. Toggle **Notebook access** to ON.

Once the secret is saved, the cell below retrieves it and writes it into `os.environ` for the SDK to pick up automatically. Your key never appears in the notebook output.

> **If you are running locally** (not on Colab), skip this cell. Set the environment variable in your terminal before launching Jupyter:
> ```bash
> export OPENAI_API_KEY=sk-...
> ```

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("✅ API key loaded from Colab Secrets.")

## Cell 3 — Import `Agent` and `Runner`

This is where the SDK's design philosophy becomes immediately visible.

Two imports. That is the entire surface area needed for a basic agent run:

- **`Agent`** — the primitive we introduced in Lecture 1.3. An `Agent` carries its instructions (system prompt), an optional model, optional tools, and optional guardrails. It is a configuration object — it does not run itself.
- **`Runner`** — the execution engine introduced in Lecture 1.4. The `Runner` drives the agent loop: it sends input to the model, processes the response, runs any tool calls, and decides whether to loop again or return a final result.

There is no framework boilerplate, no base class to inherit from, no decorator stack to set up. Just two clean imports.

In [ ]:
# Import Runner and Agent

## Cell 4 — Define the Agent

An `Agent` object is the SDK's concrete realisation of the **Agent primitive** from Lecture 1.3.

At minimum, it takes two parameters:

| Parameter | Type | What it does |
|---|---|---|
| `name` | `str` | A human-readable label for this agent. Appears in traces so you can identify which agent handled which step of a run. |
| `instructions` | `str` | The **system prompt**. This is the single most powerful parameter on the `Agent` object — it shapes everything: tone, behaviour, scope, and persona. |

Notice what is **not** here:
- No `model` parameter — the SDK picks a sensible default. Model selection is covered in depth in Section 2.
- No `tools` — tools are covered in Section 3.
- No `guardrails` — covered in Section 5.

Three lines. One agent. Let's define it.

In [ ]:
# Define the Agent. Give it a name and instructions

## Cell 5 — Run the Agent with `await Runner.run()`

The `Runner` class has three execution methods:

| Method | When to use |
|---|---|
| `Runner.run()` | The standard async method. Use this in Jupyter, Colab, FastAPI, or any async context. |
| `Runner.run_sync()` | Synchronous wrapper — only works when **no** event loop is already running. Not suitable for Jupyter or Colab. |
| `Runner.run_streamed()` | When you want to stream tokens or events in real time as they are produced. |

Google Colab (and Jupyter generally) runs its own asyncio event loop in the background. Calling `Runner.run_sync()` in this environment raises a `RuntimeError` because it tries to start a second event loop on top of the existing one. The correct approach here is `await Runner.run()` — Colab supports top-level `await` natively, so no wrapper is needed.

`Runner.run()` takes two positional arguments:
1. The **agent** to run.
2. The **input** — a plain string (treated as a user message) or a list of structured input items.

The async and streaming variants are covered in depth in Section 4. For now, `await Runner.run()` is exactly what we need.

This cell makes a **live API call** to OpenAI. Run it and wait a moment.

In [ ]:
# Run the agent with a prompt

## Cell 6 — Print `final_output`

`Runner.run()` returns a `RunResult` object. The most important property on it is `final_output` — the text the model produced at the end of the agent loop.

When the agent has no `output_type` defined (as is the case here), `final_output` is a plain `str`. When the agent uses a Pydantic `output_type`, `final_output` will be a typed object — that is covered in Section 2.

Run this cell. The haiku prints. **This is your first agent.**

In [ ]:
# Print the result

## Cell 7 — Inspect the `RunResult` Object

`result` is not just a string. It is a rich `RunResult` object that carries metadata about everything that happened during the run.

Here are three of its most useful properties:

| Property | Type | What it contains |
|---|---|---|
| `final_output` | `str` (here) | The text the model produced as its final answer. |
| `last_agent` | `Agent` | The agent that completed the run. In a multi-agent workflow with handoffs, this may not be the agent you started with. |
| `context_wrapper.usage.total_tokens` | `int` | Total tokens consumed across all model calls in the run — useful for cost tracking. |

This is just a preview. **Lecture 1.7** unpacks `RunResult` completely — including `new_items`, `input_items`, and usage metadata in depth.

For now, the key insight is: `Runner.run()` returns far more than plain text. Everything the SDK tracked during the run is available on this object.

In [ ]:
print("Final output:", result.final_output)
print("Last agent:", result.last_agent.name)
print("Total tokens used:", result.context_wrapper.usage.total_tokens)

## Cell 8 — Change the Instructions: Show the Agent is Configurable

The `Agent` object is nothing but configuration. The model underneath is the same — what changes is the **instructions**.

Let's define a second agent with a very different persona and ask it the same question. This demonstrates the most fundamental truth about working with agents:

> **The model is a substrate. The instructions are everything.**

Everything the model knows about how to behave — its tone, its scope, its persona, its constraints — comes from `instructions`. A single parameter change produces a completely different agent.

This is exactly what Section 2 is about: understanding agent configuration deeply. How to write effective instructions, how to inject runtime context dynamically, how to clone agents for persona variants. The Pirate Assistant below is a playful illustration of a principle that becomes critically important as your systems grow in complexity.

In [ ]:
agent2 = Agent(
    name="Pirate Assistant",
    instructions=(
        "You are a helpful assistant who always "
        "responds like a pirate."
    ),
)

result2 = await Runner.run(
    agent2,
    "What is recursion in programming?",
)

print(result2.final_output)